# 第 7 周练习：QLoRA 微调 LLaMA 3.1 8B（商品价格预测）

## 练习目标（理念）

在 **Google Colab** 一类 GPU 环境里，加载已经用 **QLoRA** 微调好的 **LLaMA 3.1 8B** 适配器，对商品描述做**价格预测**，并用自定义评估框架量误差与命中率。

本笔记本侧重**推理与评估**（加载基座 + PEFT 适配器），而不是从零开始训练。

## 和第 7 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 4-bit 量化（Quantization） | `BitsAndBytesConfig` + NF4，把约 32GB 显存需求压到约 5–6GB |
| QLoRA / PEFT 适配器 | `PeftModel.from_pretrained` 挂载 `ed-donner/pricer-...` |
| 因果语言模型推理 | 取下一个 token 的 logits，做 Top-K 加权平均得到价格 |
| 评估指标 | 美元误差、RMSLE、绿灯命中率 + 散点图 |

## 怎么跑

1. 在带 GPU 的运行时（如 Colab）从上到下执行
2. 配置 HuggingFace 令牌：`userdata.get('HF_TOKEN')`（需能访问 Llama 门禁模型）
3. 先装依赖 → 登录 → 加载数据与量化模型 → 挂适配器 → 跑 `Tester.test(...)`


In [ ]:
# ========== 环境自检：Python / PyTorch / CUDA / GPU 名称 ==========

# 标准库 sys：读解释器版本信息
import sys
# 打印当前内核的 Python 版本，确认环境是否符合预期
print(f"Python: {sys.version}")

# 导入 PyTorch：后面张量运算、模型加载与 CUDA 推理都依赖它
import torch
# 打印 PyTorch 版本，便于排查与 transformers / bitsandbytes 的兼容性
print(f"PyTorch: {torch.__version__}")
# 是否检测到可用 CUDA（有 GPU 且驱动正常时为 True）
print(f"CUDA Available: {torch.cuda.is_available()}")
# 当前 PyTorch 编译/绑定的 CUDA 版本号
print(f"CUDA Version: {torch.version.cuda}")
# 第 0 号 GPU 的可读名称（如 T4、A100）；无 GPU 时这里会报错
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ========== 安装依赖：transformers / peft / bitsandbytes 等（版本钉死）==========
# Jupyter 魔法 !pip：在当前内核环境安装；包名与版本约束保持原样，勿改译
!pip install -q --upgrade transformers==4.48.3 accelerate==1.3.0 datasets==3.2.0
!pip install -q --upgrade peft==0.14.0 trl==0.14.0 bitsandbytes==0.46.0
!pip install -q --upgrade matplotlib scipy scikit-learn
!pip install -q --upgrade "huggingface_hub<1.0,>=0.24.0"
# 再升一次 bitsandbytes：与上一行并存，保持作者原安装顺序
!pip install -q --upgrade bitsandbytes


## 环境准备

导入本练习要用的库，并开启 Jupyter 内联绘图（`%matplotlib inline`），为后面的量化配置、模型加载与评估图做准备。


In [ ]:
# ========== 导入：量化推理、数据集、PEFT、绘图所需工具 ==========

# 标准库 os：环境与路径相关（本格主要占位导入，与原逻辑一致）
import os
# 标准库 re：正则解析模型输出里的价格数字
import re
# 标准库 math：评估时算对数误差、开方（RMSLE）
import math
# PyTorch 主模块：张量、设备、推理
import torch
# 函数式接口 F：对本格后续 Softmax 等运算
import torch.nn.functional as F
# matplotlib：画「真值 vs 预测」散点图
import matplotlib.pyplot as plt
# tqdm：评估循环进度条
from tqdm import tqdm
# HuggingFace Hub 登录：用 token 拉取门禁模型与数据集
from huggingface_hub import login
# transformers：因果 LM、分词器、4-bit 配置、固定随机种子
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
# datasets：从 Hub 加载 pricer 数据集
from datasets import load_dataset
# peft：把 LoRA/QLoRA 适配器叠到基座模型上
from peft import PeftModel

# Jupyter 行魔法：图直接嵌在笔记本输出里（不是普通 Python，勿改）
%matplotlib inline


In [ ]:
# ========== 常量：基座模型、数据集、微调适配器与评估超参 ==========

# 基座因果 LM：Meta Llama 3.1 8B（需 HF 授权）
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
# 价格预测数据集（HuggingFace Hub 上的 id）
DATASET_NAME = "ed-donner/pricer-data"
# 已发布的 PEFT 适配器仓库名
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39"
# 适配器某次提交的 revision（git commit hash），保证可复现
REVISION = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

# Top-K：推理时取概率最高的 K 个下一 token 做加权平均
TOP_K = 3
# 评估样本数上限（与 Tester 默认 size 对齐）
TEST_SIZE = 250

# ANSI 颜色码：在终端/笔记本里给评估行上色
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
# 颜色名 → ANSI 码：供 Tester 打印时查表
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


In [ ]:
# ========== HuggingFace 登录：从 Colab secrets 取 HF_TOKEN ==========

# Colab 专用 userdata：读取笔记本 secrets，避免把 token 写进源码
from google.colab import userdata
# 取出名为 HF_TOKEN 的密钥字符串
hf_token = userdata.get('HF_TOKEN')

# 登录 HuggingFace Hub；add_to_git_credential=True 便于后续 git/push 复用
login(hf_token, add_to_git_credential=True)
# 提示登录成功（展示文案保持原样）
print("Successfully authenticated with HuggingFace")


## 加载数据

从 HuggingFace Hub 拉取 `ed-donner/pricer-data`，拆出 **train / test**。先看一条测试样例：文本前缀 + 真值价格（ground truth），建立对数据形态的直觉。


In [ ]:
# ========== 加载数据集：打印 train / test 规模 ==========

# 提示正在加载的数据集 id（字符串保持原样）
print(f"Loading dataset: {DATASET_NAME}")
# 从 Hub 下载/缓存整个 DatasetDict
dataset = load_dataset(DATASET_NAME)
# 训练集划分
train = dataset['train']
# 测试集划分（后面评估用）
test = dataset['test']

# 成功提示与样本量（千位分隔符便于阅读）
print(f"\nDataset loaded successfully:")
print(f"  Training examples: {len(train):,}")
print(f"  Test examples: {len(test):,}")


In [ ]:
# ========== 看一条测试样例：文本截断 + 真值价格 ==========

# 说明下面打印的是测试集样例
print("Sample test example:\n")
# 取测试集第 0 条（dict，含 text / price 等字段）
sample = test[0]
# 只打印 text 前 200 字符，避免刷屏
print(f"Text: {sample['text'][:200]}...")
# 真值价格，保留两位小数
print(f"\nGround truth price: ${sample['price']:.2f}")


## 量化与模型加载

用 **4-bit NF4**（`BitsAndBytesConfig`）加载 LLaMA 3.1 8B：双重量化 + `bfloat16` 计算类型，把显存占用从大约 **32GB** 降到大约 **5–6GB**，才能在消费级 / Colab T4 一类卡上跑起来。


In [ ]:
# ========== 4-bit 量化配置：NF4 + 双重量化 + bf16 计算 ==========

# BitsAndBytesConfig：告诉 from_pretrained 如何做 bitsandbytes 量化
quant_config = BitsAndBytesConfig(
    # 以 4-bit 权重加载，大幅省显存
    load_in_4bit=True,
    # 双重量化：再压缩量化常数，进一步省显存
    bnb_4bit_use_double_quant=True,
    # 前向计算用 bfloat16（在支持的 GPU 上更稳）
    bnb_4bit_compute_dtype=torch.bfloat16,
    # 量化类型 NF4（NormalFloat4，QLoRA 常用）
    bnb_4bit_quant_type="nf4"
)

# 提示当前采用的量化方案（文案保持原样）
print("Using 4-bit NF4 quantization")


In [ ]:
# ========== 加载分词器与基座因果 LM（套用上面的量化配置）==========

# 提示正在加载的基座模型 id
print(f"Loading base model: {BASE_MODEL}")

# 从 Hub 拉分词器；trust_remote_code=True 允许仓库自定义代码（与原参数一致）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# Llama 常无独立 pad_token：用 eos 充当，避免 padding 报错
tokenizer.pad_token = tokenizer.eos_token
# 右侧 padding：因果 LM 训练/批处理时的常见设定
tokenizer.padding_side = "right"

# 加载基座模型：量化配置 + 自动设备映射（多卡/单卡由 accelerate 处理）
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置里的 pad_token_id 与分词器对齐，减少警告
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 打印量化后大致显存占用（字节 / 1e9 → GB）
print(f"Base model loaded - Memory: {base_model.get_memory_footprint() / 1e9:.2f} GB")


## 加载 PEFT 适配器

把已经训练好的 **LoRA / QLoRA 适配器**叠到量化基座上。`revision` 钉死某次提交，保证和作者实验同一套权重。


In [ ]:
# ========== 挂载微调适配器：PeftModel.from_pretrained ==========

# 提示适配器仓库名
print(f"Loading fine-tuned adapters: {FINETUNED_MODEL}")
# 提示 revision（commit），便于对照复现
print(f"Revision: {REVISION}")

# 在 base_model 上加载 PEFT 权重；revision 指定确切版本
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)

# 打印叠适配器后的总显存脚印
print(f"Fine-tuned model ready - Total memory: {fine_tuned_model.get_memory_footprint() / 1e9:.2f} GB")


In [ ]:
# ========== 工具函数：从生成文本里抽出 Price is $ 后的数字 ==========

def extract_price(text):
    # 只有包含约定前缀时才解析，否则返回 0.0
    if "Price is $" in text:
        # 取前缀右侧正文
        content = text.split("Price is $")[1]
        # 去掉千分位逗号与多余美元符，方便正则
        content = content.replace(',', '').replace('$', '')
        # 抓第一个整数/小数（可带正负号）
        match = re.search(r"[-+]?\d*\.?\d+", content)
        # 匹配成功则转 float，否则 0.0
        return float(match.group()) if match else 0.0
    return 0.0


In [ ]:
# ========== 冒烟测试 extract_price：普通价 / 千分位 / 夹杂文字 ==========

# 三组字符串：覆盖常见与「脏」输出
test_cases = [
    "Price is $24.99",
    "Price is $1,234.50",
    "Price is $a fabulous 899.99 or so"
]

# 逐条解析并打印「原文 -> 解析结果」
for test in test_cases:
    result = extract_price(test)
    print(f"{test} -> ${result:.2f}")


## 预测函数

**Top-K 加权平均**：对「下一个 token」概率最高的 K 个候选，若能解码成正数价格，则按概率加权平均，得到一个连续价格估计（比只取 argmax 一个 token 更稳一点）。


In [ ]:
# ========== advanced_predict：Top-K 下一 token → 概率加权平均价格 ==========

def advanced_predict(prompt, top_k=TOP_K):
    # 固定种子，使同 prompt 的采样/数值路径更可复现
    set_seed(42)
    # 分词为 token id 张量，并放到 CUDA
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    # 全 1 attention mask：本格未做真实 padding，形状与 inputs 一致即可
    attention_mask = torch.ones(inputs.shape, device="cuda")

    # 推理不建梯度，省显存
    with torch.no_grad():
        # 前向：取整段序列的 logits
        outputs = fine_tuned_model(inputs, attention_mask=attention_mask)
        # 只要最后一个位置的 logits，并搬回 CPU 做后续概率运算
        next_token_logits = outputs.logits[:, -1, :].to('cpu')

    # Softmax → 下一 token 概率分布
    next_token_probs = F.softmax(next_token_logits, dim=-1)
    # 取概率最高的 top_k 个：值与对应 token id
    top_probs, top_token_ids = next_token_probs.topk(top_k)

    # 收集可解析为正数的价格及其概率权重
    prices, weights = [], []

    for i in range(top_k):
        # 把第 i 个高概率 token 解码成字符串
        predicted_token = tokenizer.decode(top_token_ids[0][i])
        # 对应概率
        probability = top_probs[0][i]

        try:
            # 尝试把解码串当成数字价格
            price = float(predicted_token)
            if price > 0:
                prices.append(price)
                weights.append(probability)
        except ValueError:
            # 非数字 token（如标点/词）直接跳过
            continue

    # 若 K 个里没有有效正数，返回 0.0
    if not prices:
        return 0.0

    # 归一化权重后做加权平均
    total_weight = sum(weights)
    weighted_avg = sum(p * w / total_weight for p, w in zip(prices, weights))

    # 若结果是 0 维张量则 .item() 成 Python float
    return weighted_avg.item()


## 评估框架

自定义 `Tester` 类，在测试集上批量跑预测并汇总：

- **美元误差**：`|预测 - 真值|`
- **RMSLE**：均方根对数误差（更关注相对误差）
- **命中率**：绿灯比例——误差 `< $40` **或** `< 真值的 20%`

最后画「真值 vs 预测」散点图，绿 / 橙 / 红表示误差档位。


In [ ]:
# ========== Tester：批量评估 + 上色打印 + 散点图 + 汇总指标 ==========

class Tester:

    def __init__(self, predictor, data, title=None, size=TEST_SIZE):
        # 预测函数：输入样例 text（本练习里是 advanced_predict）
        self.predictor = predictor
        # 数据集（需支持下标与 ["text"] / ["price"]）
        self.data = data
        # 报表标题：默认用函数名美化；也可外部传入
        self.title = title or predictor.__name__.replace("_", " ").title()
        # 实际评估条数：不超过数据长度
        self.size = min(size, len(data))
        # 累积：预测值、真值、绝对误差、SLE、颜色标签
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # 绿灯：绝对误差 < 40 或相对误差 < 20%
        if error < 40 or error / truth < 0.2:
            return "green"
        # 橙灯：绝对误差 < 80 或相对误差 < 40%
        elif error < 80 or error / truth < 0.4:
            return "orange"
        # 其余红灯
        else:
            return "red"

    def run_datapoint(self, i):
        # 取第 i 条样例
        datapoint = self.data[i]
        # 用预测器对 text 出价格
        guess = self.predictor(datapoint["text"])
        # 真值价格
        truth = datapoint["price"]
        # 绝对美元误差
        error = abs(guess - truth)

        # 对数误差：log(truth+1) - log(guess+1)，避免 log(0)
        log_error = math.log(truth + 1) - math.log(guess + 1)
        # SLE = 对数误差的平方（后面再均方根得 RMSLE）
        sle = log_error ** 2

        # 按误差档位上色
        color = self.color_for(error, truth)
        # 从 text 里抽第二段当短标题（约定格式含 \n\n）
        title = datapoint["text"].split("\n\n")[1][:30] + "..."

        # 写入累积列表
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)

        # 彩色打印单条结果（ANSI + RESET）
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} | Truth: ${truth:,.2f} | Error: ${error:,.2f} | SLE: {sle:,.3f} | {title}{RESET}")

    def chart(self, title):
        # 画布尺寸
        plt.figure(figsize=(14, 10))
        # 坐标轴上限：真值与预测的全局最大
        max_val = max(max(self.truths), max(self.guesses))

        # y=x 完美预测线
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=3, alpha=0.7, label='Perfect prediction')
        # 散点：颜色对应 green/orange/red 字符串（matplotlib 可识别）
        plt.scatter(self.truths, self.guesses, s=20, c=self.colors, alpha=0.6)

        # 轴标签与范围、标题、网格、图例
        plt.xlabel('Ground Truth Price ($)', fontsize=12)
        plt.ylabel('Model Prediction ($)', fontsize=12)
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    def report(self):
        # 平均美元误差
        average_error = sum(self.errors) / self.size
        # RMSLE = sqrt(mean SLE)
        rmsle = math.sqrt(sum(self.sles) / self.size)
        # 绿灯条数与命中率百分比
        hits = sum(1 for color in self.colors if color == "green")
        hit_rate = hits / self.size * 100

        # 图标题里嵌入三项核心指标
        title = f"{self.title} | Avg Error: ${average_error:,.2f} | RMSLE: {rmsle:.3f} | Hit Rate: {hit_rate:.1f}%"

        # 文本汇总（英文标题保持原样）
        print(f"\n{'='*80}")
        print(f"EVALUATION SUMMARY")
        print(f"{'='*80}")
        print(f"Model: {self.title}")
        print(f"Test Size: {self.size}")
        print(f"Average Dollar Error: ${average_error:,.2f}")
        print(f"RMSLE: {rmsle:.4f}")
        print(f"Hit Rate (Green): {hit_rate:.2f}% ({hits}/{self.size})")
        print(f"{'='*80}\n")

        # 画散点图
        self.chart(title)

    def run(self):
        # 提示评估规模
        print(f"Running evaluation on {self.size} examples...\n")
        # 带进度条逐条评估
        for i in tqdm(range(self.size), desc="Evaluating"):
            self.run_datapoint(i)
        # 汇总 + 作图
        self.report()

    @classmethod
    def test(cls, function, data, **kwargs):
        # 语法糖：构造实例并立刻 run()
        cls(function, data, **kwargs).run()


In [ ]:
# ========== 再次加载数据集：与前面加载格等价，保证 test 可用 ==========

# 提示数据集 id
print(f"Loading dataset: {DATASET_NAME}")
# 重新 load（若内核状态丢了可只跑本格恢复）
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

print(f"\nDataset loaded successfully:")
print(f"  Training examples: {len(train):,}")
print(f"  Test examples: {len(test):,}")


## 运行评估

调用 `Tester.test`，在测试集上用 `advanced_predict` 跑一遍（默认最多 `TEST_SIZE` 条），查看平均误差、RMSLE、命中率与散点图。


In [ ]:
# ========== 启动评估：QLoRA 微调后的 LLaMA 3.1 8B ==========
# function=advanced_predict；data=test；title 仅用于报表展示
Tester.test(advanced_predict, test, title="LLaMA 3.1 8B QLoRA (400K)")
